In [2]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
import warnings
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

warnings.filterwarnings("ignore")
secrets = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")
login(hf_token)

!pip install wandb -q
import wandb
wandb_token = secrets.get_secret("WANDB")
wandb.login(key=wandb_token)

# For Colab setup
'''from google.colab import userdata
from huggingface_hub import login, snapshot_download

login(userdata.get('HF_TOKEN'))  # login first, before snapshot_download'''

'''try:
    snapshot_download(
        repo_id="alexdimmock/wav2vec2-basque-10h",
        local_dir="/content/wav2vec2-basque-10h"
    )
    print("Checkpoint restored from Hub")
except Exception as e:
    print(f"No checkpoint found, starting fresh: {e}")'''

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: alex_dimmock (alex_dimmock-nas) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


'try:\n    snapshot_download(\n        repo_id="alexdimmock/wav2vec2-basque-10h",\n        local_dir="/content/wav2vec2-basque-10h"\n    )\n    print("Checkpoint restored from Hub")\nexcept Exception as e:\n    print(f"No checkpoint found, starting fresh: {e}")'

In [4]:
!pip install transformers datasets evaluate jiwer -q

from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor, TrainingArguments, Trainer
from datasets import load_dataset, Dataset, Audio, get_dataset_config_names, get_dataset_split_names
from dataclasses import dataclass
from evaluate import load as load_metric
import os, torch
import unicodedata
import re
import numpy as np

# CPU Fallback
'''os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
'''
# Load pretrained characters from basque fine-tune for token characters
processor = Wav2Vec2Processor.from_pretrained("stefan-it/wav2vec2-large-xlsr-53-basque")

print("\nBasque token vocabulary:")
print(processor.tokenizer.get_vocab())

# Load base facebook model to fine-tune
ssl_model = Wav2Vec2ForCTC.from_pretrained("facebook/wav2vec2-large-xlsr-53", vocab_size=len(processor.tokenizer))

ssl_model.freeze_feature_encoder()

# Text cleaner
def clean_text(t):
    t = t.lower()
    t = unicodedata.normalize("NFKC", t)
    t = re.sub(r"[^\w\sñíáéóúü]", "", t)  # keep letters only
    t = re.sub(r"\s+", " ", t).strip()

    return t

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 59.4 MB/s eta 0:00:0000:01


preprocessor_config.json:   0%|          | 0.00/158 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

...

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-large-xlsr-53
Key                          | Status     | 
-----------------------------+------------+-
quantizer.codevectors        | UNEXPECTED | 
quantizer.weight_proj.bias   | UNEXPECTED | 
project_hid.weight           | UNEXPECTED | 
project_q.bias               | UNEXPECTED | 
quantizer.weight_proj.weight | UNEXPECTED | 
project_hid.bias             | UNEXPECTED | 
project_q.weight             | UNEXPECTED | 
lm_head.bias                 | MISSING    | 
lm_head.weight               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
def preprocess(sample):
    '''
    Preprocess: takes a sample, separates the audio, sampling rate, input and labels
    Args: sample
    Returns: dict with input values and labels
    '''
    audio = sample["audio"]["array"]
    sr = sample["audio"]["sampling_rate"]

    inputs = processor(audio, sampling_rate=sr)
    
    text = clean_text(sample["sentence"])
    labels = processor(text=[text]).input_ids[0]
    
    return {
        "input_values": inputs.input_values[0],
        "labels": labels
    }

In [5]:
'''### TO DETERMINE NUMBER OF SAMPLES REQUIRED FOR NUMBER OF HOURS

ds = load_dataset("HiTZ/composite_corpus_eu_v2.1", split="train", streaming=True)

total_duration = 0
count = 0
for sample in ds:
    total_duration += len(sample["audio"]["array"]) / sample["audio"]["sampling_rate"]
    count += 1
    if count % 500 == 0:
        print(f"{count} samples, {total_duration/3600:.2f} hours so far")
    if total_duration >= 50 * 3600:  # stop at 10 hours
        break

print(f"50 hours reached at sample {count}")'''

README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/150 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/150 [00:00<?, ?it/s]

500 samples, 0.84 hours so far
1000 samples, 1.67 hours so far
1500 samples, 2.49 hours so far
2000 samples, 3.33 hours so far
2500 samples, 4.16 hours so far
3000 samples, 4.98 hours so far
3500 samples, 5.80 hours so far
4000 samples, 6.63 hours so far
4500 samples, 7.45 hours so far
5000 samples, 8.29 hours so far
5500 samples, 9.14 hours so far
6000 samples, 9.98 hours so far
6500 samples, 10.80 hours so far
7000 samples, 11.63 hours so far
7500 samples, 12.43 hours so far
8000 samples, 13.27 hours so far
8500 samples, 14.08 hours so far
9000 samples, 14.88 hours so far
9500 samples, 15.71 hours so far
10000 samples, 16.51 hours so far
10500 samples, 17.34 hours so far
11000 samples, 18.16 hours so far
11500 samples, 18.95 hours so far
12000 samples, 19.76 hours so far
12500 samples, 20.58 hours so far
13000 samples, 21.39 hours so far
13500 samples, 22.18 hours so far
14000 samples, 23.00 hours so far
14500 samples, 23.84 hours so far
15000 samples, 24.65 hours so far
15500 sample

In [ ]:
'''# Load 10h dataset
raw = load_dataset("HiTZ/composite_corpus_eu_v2.1", split="train", streaming=True).take(6000)
train_list = [preprocess(s) for s in raw]
cv_train_10h = Dataset.from_list(train_list)'''

# Load 50h dataset
ds_50 = load_dataset("HiTZ/composite_corpus_eu_v2.1", split="train", streaming=True).take(30500)
train_50h_list = [preprocess(s) for s in ds_50]
cv_train_50h = Dataset.from_list(train_50h_list)

# Validation set
cv_dev_raw = load_dataset("HiTZ/composite_corpus_eu_v2.1", split="dev_cv", streaming=True).take(500)
cv_dev = []
for i, sample in enumerate(cv_dev_raw.map(preprocess)):
    cv_dev.append(sample)
    if i % 50 == 0:
        print(f"Loaded {i} val samples")
print(f"Done: {len(cv_dev)} val samples")

@dataclass
class CTCDataCollator:
    processor: Wav2Vec2Processor

    def __call__(self, features):
        # Pad input_values
        input_features = [
            {"input_values": f["input_values"]}
            for f in features]
            
        batch = self.processor.pad(input_features, padding=True, return_tensors="pt")

        # Pad labels with PAD token
        label_features = [f["labels"] for f in features]
        labels_batch = self.processor.tokenizer.pad(
            {"input_ids": label_features}, padding=True, return_tensors="pt"
        )
        # Replace PAD token id with -100 so loss ignores it
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch["input_ids"] == self.processor.tokenizer.pad_token_id, -100
        )
        batch["labels"] = labels
        return batch

data_collator = CTCDataCollator(processor=processor)

from torch.utils.data import DataLoader
train_dataloader = DataLoader(
    cv_train_50h,
    batch_size=4,
    collate_fn=data_collator
)

In [4]:
from huggingface_hub import snapshot_download

path = snapshot_download(
    repo_id="alexdimmock/wav2vec2-basque-10h",
    local_dir="/kaggle/working/checkpoint",
    token=hf_token,
    ignore_patterns=["optimizer.pt"]  # skip the 2.49GB optimizer, we don't need it to resume
)
print(path)
print(os.listdir(path))

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

/kaggle/working/checkpoint
['.gitattributes', 'training_args.bin', 'model.safetensors', 'last-checkpoint', 'config.json', '.cache', 'preprocessor_config.json']


In [5]:
print(os.listdir("/kaggle/working/checkpoint/last-checkpoint"))

['trainer_state.json', 'scheduler.pt', 'rng_state.pth', 'training_args.bin', 'optimizer.pt', 'model.safetensors', 'config.json', 'preprocessor_config.json']


In [6]:
# Training arguments
training_args = TrainingArguments(
    output_dir="/kaggle/working/wav2vec2-basque-50h",
    report_to="wandb",
    run_name="basque-50h-run1",
    per_device_train_batch_size=2,
    eval_strategy="epoch",
    bf16=True,
    learning_rate=5e-4,
    warmup_steps=600,
    logging_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=20,
    save_total_limit=2,
    push_to_hub=True,
    hub_model_id="alexdimmock/wav2vec2-basque-50h",
    hub_strategy="checkpoint",
    hub_token=hf_token
    )

wer_metric = load_metric("wer")

def compute_metrics(pred):
    logits = pred.predictions
    pred_ids = np.argmax(logits, axis=-1)

    label_ids = pred.label_ids.copy()
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.batch_decode(pred_ids, group_tokens=True)
    label_str = processor.batch_decode(label_ids, group_tokens=False)

    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}
    
trainer = Trainer(
    model=ssl_model,
    data_collator=data_collator,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=cv_train_50h,
    eval_dataset=cv_dev,
    processing_class=processor.feature_extractor
)

trainer.train()

'''# Exit file
os._exit(0)'''

Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
'''### DEBUG ON SMALL OVERFITTED DATASET

from datasets import load_dataset, Dataset

raw = load_dataset("HiTZ/composite_corpus_eu_v2.1", split="train", streaming=True).take(500)
train_list = [preprocess(s) for s in raw]
cv_train_10h = Dataset.from_list(train_list)

# Quick sanity check: print what labels actually look like now
print("labels sample:", cv_train_10h[0]["labels"])
print("decoded:", processor.tokenizer.decode(cv_train_10h[0]["labels"]))

training_args = TrainingArguments(
    output_dir="./debug-overfit",
    per_device_train_batch_size=2,
    learning_rate=4e-4,   # was 1e-4
    max_steps=5000,       # was 200
    logging_steps=50,
    eval_strategy="no",
    save_strategy="steps",
    bf16=True,
)

trainer = Trainer(
    model=ssl_model,
    args=training_args,
    train_dataset=cv_train_10h,
    data_collator=data_collator,
    processing_class=processor,
)

trainer.train()

# Check a prediction after training
import torch, numpy as np
sample = cv_train_10h[10]
input_tensor = torch.tensor([sample["input_values"]]).to(ssl_model.device)
with torch.no_grad():
    logits = ssl_model(input_tensor).logits
pred_ids = logits.argmax(-1)
print("PRED:", processor.batch_decode(pred_ids)[0])
print("TRUE:", processor.tokenizer.decode(sample["labels"]))'''



'''try wer function on debug code if the above is fucked'''

In [ ]:
import os
for root, dirs, files in os.walk("/kaggle/working/last-checkpoint"):
    for f in files:
        print(os.path.join(root, f))

In [ ]:
import os
print(os.listdir("/kaggle/working"))

In [ ]:
os.system("df -h /kaggle/working")
